In [ ]:
# Public-release setup: run from any working directory.
from pathlib import Path
import sys

def find_release_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "docs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the public release directory.")

PROJECT_ROOT = find_release_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PROJECT_ROOT / "results"  # User-supplied artifacts; not included in the release.
FIGURES_ROOT = PROJECT_ROOT / "figures"


# Step1+ Light SFT Adaptation and Section 3 Validation

This notebook analyzes the small exploratory experiment requested in `codex_jds/step1plus_section3_and_appendixB2.md`.

It reads saved CSV/JSON outputs only. It does not load models or rerun validation.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebook_for_paper" else CWD
RESULT_DIR = PROJECT_ROOT / "results" / "step1plus_sft_section3_qwen25_15b_300steps_train10_310"
FIGURE_DIR = PROJECT_ROOT / "figures" / "step1plus_sft_section3"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_ORDER = ["base", "mid", "final"]
COMPARISONS = {
    "A_actual_vs_exact": ("delta_logp", "first_order_exact"),
    "B_exact_vs_approx": ("first_order_exact", "approx"),
    "C_actual_vs_approx": ("delta_logp", "approx"),
}
CHECKPOINT_LABELS = {"base": "0", "mid": "50", "final": "100"}
COMPARISON_LABELS = {
    "A_actual_vs_exact": "Actual vs exact",
    "B_exact_vs_approx": "Exact vs CH1+CH2",
    "C_actual_vs_approx": "Actual vs CH1+CH2",
}
METRIC_LABELS = {
    "pearson": "Pearson",
    "spearman_signed": "Signed Spearman",
    "spearman_abs": "Abs Spearman",
    "sign_agreement": "Sign agreement",
    "slope_y_on_x_through_origin": "Slope",
}
COLORS = {
    "A_actual_vs_exact": "#2f6f9f",
    "B_exact_vs_approx": "#c75146",
    "C_actual_vs_approx": "#5a8f3d",
    "base": "#4d4d4d",
    "mid": "#2f6f9f",
    "final": "#c75146",
}

SAVE_PNG = True
SAVE_PDF = True
SCATTER_ALPHA = 0.35
SCATTER_SIZE = 14

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RESULT_DIR   = {RESULT_DIR}")
print(f"FIGURE_DIR   = {FIGURE_DIR}")

## Load Data

In [ ]:
paths = {
    "metrics": RESULT_DIR / "checkpoint_comparison_metrics.csv",
    "support": RESULT_DIR / "checkpoint_supporting_diagnostics.csv",
    "loss": RESULT_DIR / "sft_loss_curve.csv",
    "pairs": RESULT_DIR / "all_checkpoint_section3_pairs.csv",
    "run_config": RESULT_DIR / "run_config.json",
    "checkpoint_summary": RESULT_DIR / "checkpoint_summary.json",
}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

metrics_df = pd.read_csv(paths["metrics"])
support_df = pd.read_csv(paths["support"])
loss_df = pd.read_csv(paths["loss"])
pairs_df = pd.read_csv(paths["pairs"])
run_config = json.loads(paths["run_config"].read_text())
checkpoint_summary = json.loads(paths["checkpoint_summary"].read_text())

metrics_df["checkpoint"] = pd.Categorical(metrics_df["checkpoint"], CHECKPOINT_ORDER, ordered=True)
support_df["checkpoint"] = pd.Categorical(support_df["checkpoint"], CHECKPOINT_ORDER, ordered=True)
pairs_df["checkpoint"] = pd.Categorical(pairs_df["checkpoint"], CHECKPOINT_ORDER, ordered=True)

print("metrics", metrics_df.shape)
print("support", support_df.shape)
print("loss", loss_df.shape)
print("pairs", pairs_df.shape)
print(json.dumps({k: run_config.get(k) for k in ["model_name_or_path", "adapt_split", "update_split", "observe_split", "adapt_lr", "validation_lr", "num_pairs", "num_adapt_samples"]}, indent=2))

## Helpers

In [ ]:
def save_figure(fig, stem):
    saved = []
    if SAVE_PNG:
        p = FIGURE_DIR / f"{stem}.png"
        fig.savefig(p, dpi=300, bbox_inches="tight")
        saved.append(p)
    if SAVE_PDF:
        p = FIGURE_DIR / f"{stem}.pdf"
        fig.savefig(p, bbox_inches="tight")
        saved.append(p)
    print("saved:", *saved, sep="\n  ")
    return saved


def symmetric_limits(*arrays, pad=0.06):
    vals = np.concatenate([np.asarray(a, dtype=float).ravel() for a in arrays])
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (-1, 1)
    lim = np.max(np.abs(vals))
    if lim == 0:
        lim = 1.0
    lim *= 1 + pad
    return (-lim, lim)


def corr_value(x, y, method="pearson"):
    x = pd.Series(x, dtype=float)
    y = pd.Series(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2:
        return np.nan
    if method == "pearson":
        return float(x.corr(y, method="pearson"))
    if method == "spearman":
        return float(x.rank(method="average").corr(y.rank(method="average"), method="pearson"))
    raise ValueError(method)


def annotate_corr(ax, x, y, method="pearson"):
    label = "Pearson" if method == "pearson" else "Spearman"
    r = corr_value(x, y, method)
    ax.text(
        0.04, 0.96,
        f"{label} r = {r:.3f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8.5,
        bbox={"facecolor": "white", "edgecolor": "0.85", "boxstyle": "round,pad=0.25"},
    )


def scatter_panel(ax, df, x_col, y_col, title, color, corr_method="pearson"):
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    ax.scatter(x[mask], y[mask], s=SCATTER_SIZE, alpha=SCATTER_ALPHA, color=color, marker="+")
    lo, hi = symmetric_limits(x[mask], y[mask])
    ax.plot([lo, hi], [lo, hi], color="black", lw=0.8, ls="--")
    ax.axhline(0, color="0.78", lw=0.8)
    ax.axvline(0, color="0.78", lw=0.8)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.grid(True, alpha=0.2)
    annotate_corr(ax, x[mask], y[mask], corr_method)

## SFT Loss Curve

In [ ]:
def plot_sft_loss_curve():
    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    ax.plot(loss_df["step"], loss_df["loss"], color="#2f6f9f", lw=1.3)
    for ckpt, meta in run_config["checkpoint_meta"].items():
        step = meta["step"]
        ax.axvline(step, color="0.65", lw=0.8, ls="--")
        #ax.text(step, ax.get_ylim()[1], ckpt, va="top", ha="center", fontsize=8)
    ax.set_xlabel("SFT step")
    ax.set_ylabel("Training loss")
    ax.set_title("Auxiliary GSM8K SFT loss")
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    save_figure(fig, "step1plus_sft_loss_curve")
    return fig, ax

plot_sft_loss_curve();

## Checkpoint Metric Comparison

In [ ]:
COMPARISONS_sub =  {'B_exact_vs_approx': ('first_order_exact', 'approx'),
 'C_actual_vs_approx': ('delta_logp', 'approx')}

In [ ]:
def plot_checkpoint_metrics():
    metric_names = ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]
    x = np.arange(len(CHECKPOINT_ORDER))
    fig, axes = plt.subplots(1, 4, figsize=(13.2, 3.2), sharex=True)
    for ax, metric in zip(axes, metric_names):
        for comparison in COMPARISONS_sub:
            sub = metrics_df[metrics_df["comparison"] == comparison].set_index("checkpoint").reindex(CHECKPOINT_ORDER)
            ax.plot(x, sub[metric], marker="o", lw=1.5, label=COMPARISON_LABELS[comparison], color=COLORS[comparison])
        #ax.axhline(0, color="0.78", lw=0.8)
        if metric == "sign_agreement":
            ax.axhline(0.5, color="0.86", lw=0.8, ls="--")
        ax.set_xticks(x)
        ax.set_xticklabels([CHECKPOINT_LABELS[c] for c in CHECKPOINT_ORDER])
        ax.set_title(METRIC_LABELS[metric])
        ax.set_xlabel('SFT step')
        #ax.set_ylim(-0.05 if metric == "sign_agreement" else -0.1, 1.05)
    axes[0].set_ylabel("Metric")
    axes[0].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    save_figure(fig, "step1plus_checkpoint_metric_comparison")
    return fig, axes

plot_checkpoint_metrics();

## Supporting Diagnostics

In [ ]:
def plot_supporting_diagnostics():
    cols = [
        ("mean_update_token_logp", "Mean update-token log p"),
        ("mean_update_grad_norm", "Mean update grad norm"),
        ("mean_abs_actual_delta_logp", "Mean |actual delta|"),
        ("mean_abs_first_order_exact", "Mean |first-order exact|"),
    ]
    x = np.arange(len(CHECKPOINT_ORDER))
    data = support_df.set_index("checkpoint").reindex(CHECKPOINT_ORDER)
    fig, axes = plt.subplots(1, 4, figsize=(13.2, 3.2))
    for ax, (col, title) in zip(axes, cols):
        ax.plot(x, data[col], marker="o", color="#2f6f9f", lw=1.5)
        ax.set_xticks(x)
        ax.set_xticklabels([CHECKPOINT_LABELS[c] for c in CHECKPOINT_ORDER])
        ax.set_title(title)
        ax.grid(True, alpha=0.2)
    fig.tight_layout()
    save_figure(fig, "step1plus_supporting_diagnostics")
    return fig, axes

plot_supporting_diagnostics();

## Signed Scatter Plots

In [ ]:
def plot_signed_scatters(checkpoints=("base", "final")):
    cols = [
        ("delta_logp", "first_order_exact", "Actual vs exact"),
        ("first_order_exact", "approx", "Exact vs CH1+CH2"),
        ("delta_logp", "approx", "Actual vs CH1+CH2"),
    ]
    for checkpoint in checkpoints:
        sub = pairs_df[pairs_df["checkpoint"] == checkpoint]
        fig, axes = plt.subplots(1, 3, figsize=(11.7, 3.5))
        for ax, (x_col, y_col, title) in zip(axes, cols):
            scatter_panel(ax, sub, x_col, y_col, f"{CHECKPOINT_LABELS[checkpoint]}: {title}", COLORS[str(checkpoint)], "pearson")
        fig.tight_layout()
        save_figure(fig, f"step1plus_signed_scatters_{checkpoint}")
    return None

plot_signed_scatters();

## Tables

In [ ]:
display(metrics_df)
display(support_df)
display(loss_df.describe())

## Interpretation Notes

In [ ]:
base = metrics_df[metrics_df["checkpoint"] == "base"].set_index("comparison")
final = metrics_df[metrics_df["checkpoint"] == "final"].set_index("comparison")
summary = pd.DataFrame({
    "metric": ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"],
    "A_base": [base.loc["A_actual_vs_exact", m] for m in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]],
    "A_final": [final.loc["A_actual_vs_exact", m] for m in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]],
    "B_base": [base.loc["B_exact_vs_approx", m] for m in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]],
    "B_final": [final.loc["B_exact_vs_approx", m] for m in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]],
    "C_base": [base.loc["C_actual_vs_approx", m] for m in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]],
    "C_final": [final.loc["C_actual_vs_approx", m] for m in ["pearson", "spearman_signed", "spearman_abs", "sign_agreement"]],
})
display(summary)
print("Interpretation: closest to Case B. Both Taylor/locality calibration and structural exact-vs-CH1+CH2 metrics improve after mild GSM8K adaptation, although gradients and log-prob changes shrink substantially.")